# Classify using DTW on stretched spike second deriv

In [ ]:
import os

from dtw import dtw

from workflow import load_training_dataset

In [ ]:
cut_params = {
    'signal_name': 'wideband',
    'event_name': 'spk',
    'cut_window': [-0.001, 0.002],
    'fs': 40000
}
proc_params = {
    'smooth_params': {
        'window_length': 9,
        'polyorder': 3
    },
    'dv_threshold': 0.1
}

In [ ]:
INVITRO_DATA_DIR = f'{os.environ["RAAP_INVITRO_DATA"]}/juxta'
INVIVO_DATA_DIR = f'{os.environ["RAAP_INVIVO_DATA"]}'
INVIVO_FEATURE_DIR = f'{os.environ["RAAP_INVIVO_FEATURES"]}'

In [ ]:
training_cell_names = [
    '2406241PZ_01',
    '2406242PZ_03',
    '2406263PZ_01',
    '2406282PZ_01',
    '2407031PZ_01',
    '2407031SI_gr2',
    '2407041SI_gr1'
]

In [ ]:
training_dataset = load_training_dataset(
    training_cell_names,
    INVITRO_DATA_DIR,
    cut_params=cut_params,
    proc_params=proc_params
)

In [ ]:
from transformers import (
    FirstDeriv,
    rising_phase,
    first_derivative,
    second_derivative,
    resample_first_deriv,
    resample_spike_shape,
    normalize_first_deriv,
    normalize_second_deriv
)
test_shapes = {}
for spike_type in ['AISAP', 'RAAP']:
    test_shape = training_dataset[0][spike_type][0]
    test_rising = rising_phase(test_shape, start=0, end=1.0)
    test_rising_resampled = resample_spike_shape(test_rising, 20)
    test_rising_dv1 = first_derivative(test_rising_resampled)
    test_rising_dv2 = second_derivative(test_rising_resampled)
    test_shapes[spike_type] = {
        'shape': test_rising_resampled,
        'dv1': test_rising_dv1,
        'dv2': test_rising_dv2
    }


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 1, figsize=(3, 9))
for spike_type in ['AISAP', 'RAAP']:
    for i, feature in enumerate(['shape', 'dv1', 'dv2']):
        axes[i].plot(
            test_shapes[spike_type][feature].x,
            test_shapes[spike_type][feature].y,
            label=spike_type
        )
plt.show()

In [ ]:
import numpy as np
n_training_samples = len(training_dataset)
derivs, y = [], []
for i, spike_type in enumerate(['AISAP', 'RAAP']):
    for cell_shapes in training_dataset:
        for spike in cell_shapes[spike_type]:
            deriv = second_derivative(
                resample_spike_shape(
                    rising_phase(spike, start=0.15, end=0.85), 20
                )
            )
            derivs.append(deriv)
            y.append(i)
y = np.array(y)
n_spikes = len(derivs)

In [ ]:
# from tqdm import tqdm
# D = np.zeros((n_spikes, n_spikes))
# for i in tqdm(range(n_spikes), total=n_spikes):
#     for j in range(i + 1, n_spikes):
#         alignment = dtw(
#             derivs[i].ddv,
#             derivs[j].ddv,
#             dist_method='euclidean'
#         )
#         D[i, j] = getattr(alignment, 'distance')
#         D[j, i] = D[i, j]

In [ ]:
from dtw_classifier.dataclasses import SecondDeriv


group_mean_curves = []
for yy in [0, 1]:
    group_curves = [curve for curve, label in zip(derivs, y) if label == yy]
    group_mean_curves.append(
        SecondDeriv(
            np.mean([curve.times for curve in group_curves], axis=0),
            np.mean([curve.ddv for curve in group_curves], axis=0)
        )
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 6))
for deriv, label in zip(derivs, y):
    axes[label].plot(
        group_mean_curves[label].x,
        group_mean_curves[label].y,
        color='C0' if label == 0 else 'C1',
        alpha=0.3
    )
    abs_max = np.max(np.abs(group_mean_curves[label].y))
    axes[label].set_ylim(-abs_max * 1.1, abs_max * 1.1)
plt.show()

In [ ]:
template_alignment = dtw(
    group_mean_curves[0].ddv,
    group_mean_curves[1].ddv,
    dist_method='euclidean',
    keep_internals=True
)
print(f'Template DTW distance: {getattr(template_alignment, "distance")}')

In [ ]:
template_alignment.plot(type="twoway")

In [ ]:
import pandas as pd
D_from_mean = np.zeros((n_spikes, 2))
for i, deriv in tqdm(enumerate(derivs), total=n_spikes):
    for yy in [0, 1]:
        alignment = dtw(
            deriv.ddv,
            group_mean_curves[yy].ddv,
            dist_method='euclidean'
        )
        D_from_mean[i, yy] = getattr(alignment, 'distance')

mean_dist_df = pd.DataFrame(np.stack([D_from_mean[:, 0], D_from_mean[:, 1], y], axis=1), columns=['AISAP', 'RAAP', 'Label'])
mean_dist_df['id'] = mean_dist_df.index
mean_dist_df['pref'] = (
    mean_dist_df['AISAP'] - mean_dist_df['RAAP']
) / (
    mean_dist_df['AISAP'] + mean_dist_df['RAAP']
)

In [ ]:
ais_closest_ais = mean_dist_df[mean_dist_df['Label'] == 0]['AISAP'].argmin()
ais_farthest_ais = mean_dist_df[mean_dist_df['Label'] == 0]['AISAP'].argmax()
ais_closest_ra = mean_dist_df[mean_dist_df['Label'] == 0]['RAAP'].argmin()
ais_farthest_ra = mean_dist_df[mean_dist_df['Label'] == 0]['RAAP'].argmax()

ra_closest_ais = mean_dist_df[mean_dist_df['Label'] == 1]['AISAP'].argmin()
ra_farthest_ais = mean_dist_df[mean_dist_df['Label'] == 1]['AISAP'].argmax()
ra_closest_ra = mean_dist_df[mean_dist_df['Label'] == 1]['RAAP'].argmin()
ra_farthest_ra = mean_dist_df[mean_dist_df['Label'] == 1]['RAAP'].argmax()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
colors = {
    'AISAP': 'C0',
    'RAAP': 'C1'
}
close_indices = [
    [ais_closest_ais, ais_farthest_ais],
    [ra_closest_ra, ra_farthest_ra]
]
for i, (spike_type, group_curve, members) in enumerate(zip(['AISAP', 'RAAP'], group_mean_curves, close_indices)):
    for j, idx in enumerate(members):
        axes[i, j].plot(
            group_curve.times,
            group_curve.ddv,
            color=colors[spike_type]
        )
        axes[i, j].plot(
            derivs[idx].times,
            derivs[idx].ddv,
            '--',
            color=colors[spike_type]
        )
        axes[i, j].set_title(
            f'{mean_dist_df.iloc[idx][spike_type]:.2f}'
        )

plt.show()

In [ ]:
mean_dist_df

In [ ]:
import seaborn as sns

mean_dist_df['dist_index'] = (mean_dist_df['AISAP'] - mean_dist_df['RAAP']) / (mean_dist_df['AISAP'] + mean_dist_df['RAAP'])

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
sns.scatterplot(
    data=mean_dist_df,
    x='AISAP',
    y='RAAP',
    hue='Label',
    ax=ax[0]
)
ax[0].plot(
    [0, 0.5],
    [0, 0.5],
    '--',
    color='black'
)
sns.kdeplot(
    data=mean_dist_df,
    x='dist_index',
    hue='Label',
    ax=ax[1]
)
plt.show()